# Majority-class baseline

This notebook evaluates **early versus late stages of a set** on held-out participants. Labels are proxies; performance does not establish peripheral fatigue. No models have been trained in this delivered notebook. Run the cells in order on your private Colab runtime; save the executed notebook to retain its displayed results.

The four notebooks share `modelling/core.py` so splits, selection and metrics are consistent. Outputs are restricted participant-derived results, even though IDs are pseudonymous. The majority baseline uses **no signal predictors**. The required trained baseline uses **RMS + MDF**; the other two models select from all 36 candidates using training data only.

## Portable setup

This is a clean public source notebook; no participant results are embedded.
Open it from the cloned project. If necessary set `EMG_PROJECT_ROOT` to that
project root. There is no personal Drive path or automatic Drive mount. Mount
your own private storage separately when using Colab. Restore authorized private
inputs as described in the root `README.md` (Dataset and experiment annotations) before running data cells.

Install root `requirements.txt` in the selected environment, restart that kernel,
then run cells in order. The optional installer below defaults to False. CNN
users also need compatible PyTorch; see the root README. Scientific settings,
labels, grouping and model procedures are retained; setup/path handling changes.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from importlib.metadata import version

# Set this only when the notebook is opened outside the cloned project.
PROJECT_ROOT_OVERRIDE = os.environ.get('EMG_PROJECT_ROOT')
start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
PROJECT_ROOT = next((p.resolve() for p in [start, *start.parents]
    if (p / 'project_paths.py').is_file() and (p / 'modelling/core.py').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open the notebook from the cloned project, or set EMG_PROJECT_ROOT to its root.')
sys.path.insert(0, str(PROJECT_ROOT))
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')])
    raise RuntimeError('Restart this kernel, restore INSTALL_REQUIREMENTS=False, then run from the first cell.')
for package, expected in {'numpy':'2.2.6','pandas':'2.3.2','scipy':'1.17.1','scikit-learn':'1.8.0'}.items():
    actual=version(package)
    print(package,actual)
    if actual!=expected:raise RuntimeError('Install requirements in this environment and restart the kernel.')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
probe=subprocess.run([sys.executable,'-c','import numpy, pandas, scipy.sparse, sklearn; print("Fresh-process imports OK")'],capture_output=True,text=True)
print(probe.stdout)
if probe.returncode:raise RuntimeError(probe.stderr)
print('Project:',PROJECT_ROOT)
print('Python:',sys.version.split()[0])


In [ ]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Image, Markdown
from modelling.core import DEFAULTS, MODEL_NAMES, load_data, run_model

MODEL_KEY = "majority"
DATA_DIR = PROJECT_ROOT / "modelling/data"
OUTPUT_ROOT = PROJECT_ROOT / "modelling/outputs"  # Change to private persistent storage if available.
CONFIG = dict(DEFAULTS)
# Full-cohort refit runs only AFTER the held-out evaluation; its training score is not reported.
CONFIG["final_refit"] = True
assert sklearn.__version__ == "1.8.0", "Install pinned requirements and restart the kernel."

windows, dictionary, provenance = load_data(DATA_DIR)
display(pd.DataFrame([{"model": MODEL_NAMES[MODEL_KEY], "subjects": windows.subject_id.nunique(),
    "sets": windows.recording_id.nunique(), "windows": len(windows),
    "processing_run": provenance["processing_run_id"]}]))
display(pd.Series(CONFIG, name="Frozen choices").to_frame())

## 2. Labels, eligibility and class counts

Label **0**: first third of the reviewed active interval. Label **1**: last third. These are **elapsed-time thirds**, adapting the assignment's repetition-count example because repetition markers are unavailable. The middle third and any 500 ms windows crossing a stage boundary are excluded. Hop is 250 ms. Keep both legs/all sets together in every split.

The bundle honors the saved set exclusion (`S009_REC01`), reviewed boundaries and computational/flatline/manual-artifact eligibility rules. Provisional transient/drift flags remain flags rather than automatic exclusions. MVC normalization cannot be performed without MVC references. Preprocessing is offline and can use later samples within the same recording; this does not establish real-time performance.

Earlier whole-cohort statistics and trajectories informed development. Fold-local selection prevents direct fitting on held-out subjects, but does not make this a pristine confirmatory study. The exclusion and boundary reviews also preceded validation. A new cohort would provide a stronger independent test.

### Descriptive class-imbalance audit

Report counts, percentages and the majority/minority ratio overall, by participant, leg and recording. Check every outer training/test partition and every inner training/validation partition. Counts alone need no significance test: overlapping windows are not independent observations. Equal elapsed-time thirds can yield different retained counts after window alignment and eligibility checks. The audit does not change the approved no-resampling/no-class-weighting policy.

The saved class-balance table and figure can be used in the report's dataset/label description. The actual model input remains individual windows in `model_windows.csv`; the 60 rows of `class_counts.csv` are participant × leg × class summaries.

In [ ]:
from modelling.class_balance import audit_balance
BALANCE_OUT = OUTPUT_ROOT / "class_balance" / provenance["data_sha256"][:12]
balance = audit_balance(windows, BALANCE_OUT, seed=CONFIG["seed"], inner_folds=CONFIG["inner_folds"])
for name in ["overall", "by_subject", "by_leg", "loso_partitions"]:
    display(Markdown("**" + name.replace("_", " ") + "**"))
    display(balance[name].round(3))
display(Image(filename=str(BALANCE_OUT / "class_balance.png")))
print((BALANCE_OUT / "class_balance_notes.txt").read_text(encoding="utf-8"))
display(pd.DataFrame({"feature": dictionary.feature,
    "missing_windows": windows[dictionary.feature].isna().sum().to_numpy()}))
display(Markdown("**Excluded set IDs:** " + ", ".join(provenance["excluded_recording_ids"])))

## 3. Frozen evaluation and modelling procedure

**Outer LOSO:** 15 fits/evaluations holding out one participant at a time. The trained models additionally run their inner searches; you do not run these by hand. Test labels never enter selection, imputation, scaling, regularization search or threshold choice. The same subject assignments are used in all four notebooks.

**Inner search:** three shuffled, seeded subject-grouped folds on the outer training participants. Optimize the mean of validation participants' macro-F1 values, giving participants equal weight. For both logistic models and LinearSVC search L2 regularization `C = [0.01, 0.1, 1, 10, 100]`. Selected-feature models also search maximum feature counts `[2, 3, 5]`. Ties follow deterministic parameter-grid order (smaller C, then smaller maximum feature count). No resampling/class weighting or threshold optimization. The majority baseline has no tuning.

**Selection for the proposed models:** within *each inner training split*, compute stage medians per set/feature (at least three defined windows in each stage), then average paired late–early changes across sets within each leg and equally across both legs within each participant. Require both legs and at least three contributing participants per feature. Rank by `abs(mean(participant changes) / sample SD(participant changes))`. Zero-variance or insufficient changes are unavailable. Traverse this ranking and skip features with absolute Spearman correlation ≥0.90 with an already retained feature, measured across the same training participants' changes. Unestimable correlations are flagged and that candidate is skipped. Retain up to the searched maximum; do not force extra redundant features. No p-value cutoff is used.

This redundancy rule concerns **participant changes**, not all possible predictor dependencies. Training-window correlations are also exported as diagnostics. No family quota is forced. Feature count/families may differ by fold. No held-out data decides which features are kept.

After selection: training-median imputation → training mean/SD standardization → classifier. Baseline LR fixes RMS + MDF and tunes only C. Logistic regression uses `lbfgs`; LinearSVC uses L2 squared hinge loss and `dual=False`, with `max_iter=20000`. Default prediction cutoffs remain fixed. A convergence warning fails the run rather than silently accepting an unfinished fit. Fits run serially with one numerical-library thread to limit memory use.

After evaluation, an optional final search on all participants produces a usable pipeline. This final refit provides **no additional independent test result**. Do not compare its training score with LOSO results.

## 4. Execute this model

This cell performs the actual training/evaluation. Each completed fold prints progress and saves a fitted pipeline, its split, selected features, winning inner-fold selector audits, search results and warnings. A new timestamped run directory preserves previous runs. A failed run is explicitly marked incomplete and does not replace the latest completed result.

In [ ]:
OUT = run_model(MODEL_KEY, DATA_DIR, OUTPUT_ROOT, config=CONFIG)
print("Completed run:", OUT)
run_manifest = json.loads((OUT / "run_manifest.json").read_text())
assert run_manifest["status"] == "complete"
assert run_manifest["n_outer_folds"] == windows.subject_id.nunique()

## 5. Metrics for Sections 5–6

The primary summary gives each held-out participant equal weight: mean ± **sample SD** across 15 outer folds (`ddof=1`). SD is between-participant variation, not a confidence interval. Macro precision/recall/F1 average the two classes equally; zero denominators use zero and supports are shown. Per-class F1 and Cohen's kappa are included.

ROC-AUC, trapezoidal PR-AUC and **average precision (AP)** use late stage as the positive class. PR-AUC and AP use different integration conventions and are reported separately. Linear SVM uses uncalibrated decision scores; logistic regression and the majority baseline use class-1 probabilities. AUCs are evaluated **within each held-out participant** and summarized across folds, without pooling score scales from different fitted models. Undefined AUCs remain missing and the number of defined folds is shown.

For a constant majority score ROC-AUC is 0.5, AP equals late-stage prevalence, and trapezoidal PR-AUC can be noticeably larger because of endpoint interpolation; that does not indicate discrimination. Pooled window-weighted metrics are secondary. Confusion matrices pool counts across all out-of-fold predictions, so participants with more windows contribute more counts.

In [ ]:
fold_metrics = pd.read_csv(OUT / "results.csv")
summary = pd.read_csv(OUT / "metrics_mean_sd.csv")
summary["mean_plus_minus_sd"] = summary.apply(
    lambda r: f"{r['mean']:.3f} ± {r['sd']:.3f}", axis=1)
display(summary[["metric", "mean_plus_minus_sd", "n_defined", "n_folds"]])
display(fold_metrics[["fold", "test_subject", "accuracy", "precision_macro", "recall_macro",
    "f1_macro", "f1_class_0", "f1_class_1", "cohen_kappa", "roc_auc", "pr_auc", "average_precision"]])
display(pd.read_csv(OUT / "per_class_metrics.csv"))
display(pd.read_csv(OUT / "per_class_mean_sd.csv", header=[0,1], index_col=0))
display(Markdown("**Secondary pooled window metrics:**"))
display(pd.Series(json.loads((OUT / "pooled_window_metrics_secondary.json").read_text())).to_frame("value"))

In [ ]:
for image_name in ["confusion_matrices.png", "roc_pr_by_subject.png",
                   "per_subject_f1.png", "metrics_mean_sd.png", "validation_diagram.png"]:
    display(Image(filename=str(OUT / "figures" / image_name)))

## 6. Feature selection, coefficients and bilateral results

Selection frequency describes stability under this procedure; it is **not** a significance test. Coefficients are in standardized training-feature units, conditional on correlated predictors. Their magnitudes do not establish physiological importance. The main LR versus RMS+MDF LR comparison examines feature-set choice; LR versus SVM using the same selection procedure examines the classifier, although tuning can choose different feature counts. This is not an individual-feature ablation study.

Both legs are grouped within each participant for validation. The table below compares held-out performance by leg; recordings are separate sets, so simultaneous bilateral channel relationships cannot be inferred. This table is descriptive, without additional significance testing.

In [ ]:
frequency_path = OUT / "selection_frequency.csv"
if frequency_path.exists():
    display(pd.read_csv(frequency_path))
    display(fold_metrics[["test_subject", "selected_features", "best_params", "inner_mean_subject_macro_f1"]])
else:
    print("Majority baseline: no features selected, no coefficients, no inner search.")
display(pd.read_csv(OUT / "per_subject_leg_metrics.csv"))
display(pd.read_csv(OUT / "paired_leg_f1.csv"))
oof = pd.read_csv(OUT / "out_of_fold_predictions.csv")
assert oof.window_id.is_unique and len(oof) == len(windows)
print("Exactly one held-out prediction for every eligible labelled window.")

## 7. Keep the results and use a final model later

`results.csv` in this run contains one row per outer fold. `modelling/outputs/results.csv` combines the latest completed run from each available model **only when input data, code and settings match**; after all four notebooks it should contain 60 rows. Its companion manifest identifies included runs. Do not choose features, thresholds or regularization after inspecting a held-out subject's performance and still call that subject untouched.

The final pipeline is `final_refit/pipeline.joblib`. For future recordings, first apply the same frozen preprocessing, quality rules and feature definitions, then call the pipeline's `predict` or score method. Do not refit its imputer/scaler/selector on the new participant. Metadata columns are required by the selected-feature pipeline's input schema, but only selected feature values reach the classifier. The majority baseline only needs a placeholder RMS column, which it ignores. Load only your own trusted joblib files.

There is no untouched extra test subject in this dataset: LOSO supplies held-out estimates for the development procedure. The final all-participant pipeline needs a new cohort for independent testing. Within-set elapsed-time labels also limit the claim to stage discrimination under this labelling rule.

Run the next cell to create a local results ZIP. It does not upload anything. Save/download that archive **and the executed notebook** yourself. Keep them private.

In [ ]:
import shutil
# Include the descriptive audit in this model's downloadable results.
shutil.copytree(BALANCE_OUT, OUT / "class_balance", dirs_exist_ok=True)
archive = shutil.make_archive(str(OUT.parent / (OUT.name + "_results")), "zip", root_dir=OUT)
print("Results archive:", archive)
print("Save this executed notebook separately to retain displayed outputs.")
combined = OUTPUT_ROOT / "results.csv"
if combined.exists():
    combined_results = pd.read_csv(combined)
    display(combined_results.groupby("model").size().rename("completed_outer_folds").to_frame())